In [27]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import plotly.express as px
# Import the processing module from the same folder
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields
# import processing 
# from pivottablejs import pivot_ui
G_save = True

In [28]:
ss = [
    # {'solution_folder': f"RTS-GMLC_envelope_benchmark_v2.0s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_envelope_benchmark_v2.1s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_envelope_compare_v3.0s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_envelope_compare_v3.1s", 'model_type' : 'envelope'},
    
    {'solution_folder': f"RTS-GMLC_e_reserve_benchmark_v2.0s", 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_e_reserve_benchmark_v2.1s", 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_e_reserve_compare_v3.0s", 'model_type' : 'e-reserve'},
    ]

s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    s_uc_name = 's_suc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_uc_ = load_solutions(s_uc_name, os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_uc.append(s_uc_)
    # s_ed.append(s_ed_)

    gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], solution_id = s) 

    gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], solution_id = s)

    gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

# s_uc = combine_solutions(s_uc)
# s_ed = combine_solutions(s_ed)
gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

if 'µ' in gcdi_KPI_adequacy.columns: 
#         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
    gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1.0) else x['model_type'], axis=1)
    gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if (x['model_type'] == 'envelope') & (x['µ'] == 1.0) else x['model_type'], axis=1)




../output/RTS-GMLC_envelope_benchmark_v2.1s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_envelope_benchmark_v2.1s/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_envelope_compare_v3.1s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_envelope_compare_v3.1s/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_e_reserve_benchmark_v2.0s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_e_reserve_benchmark_v2.0s/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_e_reserve_compare_v3.0s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_e_reserve_compare_v3.0s/all_gcdi_KPI_adequacy.parquet


In [29]:
# values_ = ['reserve_cost_uc','slack_reserve_up_cost_uc','slack_reserve_down_cost_uc','energy_reserve_cost_uc','slack_energy_reserve_up_cost_uc','slack_energy_reserve_down_cost_uc']
# values_ += [
#     'reserve_up_uc_MWh', 'reserve_down_uc_MWh', 'slack_reserve_up_uc_MWh', 'slack_reserve_down_uc_MWh',
#     'required_reserve_up_uc_MWh', 'required_reserve_down_uc_MWh',
#     'energy_reserve_up_uc_MWh', 'energy_reserve_down_uc_MWh', 'slack_energy_reserve_up_uc_MWh', 'slack_energy_reserve_down_uc_MWh',
#     'required_energy_reserve_up_uc_MWh', 'required_energy_reserve_down_uc_MWh'
# ]
# values_ += ['EOV',]
# values_ = [v for v in values_ if v in gcd_KPI_adequacy.columns]
# pivot = pd.pivot_table(
#     gcd_KPI_adequacy,
#     index=['day', 'solution_id'],
#     columns='model_type',
#     values=values_
# )

In [30]:
from itertools import product

days = [2,3]
scalar = []
scalar_ =pd.DataFrame()
for sol in ss:
    s = sol['solution_folder']
    for day in days:
        try:
            scalar_ = pd.read_parquet(os.path.join("..", "output", s, f'n_{day}','s_ed_scalar.parquet'))
            print(f'(ss, days):{s}, n_{day}')
            scalar_['day'] = day
            scalar_['solution_id'] = sol['solution_folder']
            scalar.append(scalar_)
        except Exception:
            pass
    
    days_str = "-".join(str(d) for d in days)
    try:
        scalar_ = pd.read_parquet(os.path.join("..", "output", s, f'n_{days_str}', 's_ed_scalar.parquet'))
        print(f'(ss, days):{s}, n_{days_str}')
        # scalar_['day'] = 0
        scalar_['solution_id'] = sol['solution_folder']
        scalar.append(scalar_)
    except Exception:
        pass

scalar = pd.concat(scalar)        

(ss, days):RTS-GMLC_envelope_benchmark_v2.1s, n_2


(ss, days):RTS-GMLC_envelope_benchmark_v2.1s, n_3
(ss, days):RTS-GMLC_envelope_compare_v3.1s, n_2
(ss, days):RTS-GMLC_envelope_compare_v3.1s, n_3
(ss, days):RTS-GMLC_e_reserve_benchmark_v2.0s, n_2
(ss, days):RTS-GMLC_e_reserve_benchmark_v2.0s, n_3
(ss, days):RTS-GMLC_e_reserve_compare_v3.0s, n_2
(ss, days):RTS-GMLC_e_reserve_compare_v3.0s, n_3


In [31]:
scalar

,objective_value,termination_status,OPEX,iteration,day,configuration,solution_id,objective_value_discrete_model,termination_status_discrete_model,primal_status_discrete_model,dual_status_discrete_model,relative_gap_discrete_model,solve_time,primal_status,dual_status
0,486029.065035,OPTIMAL,486029.065035,demand_1,2,base_ramp_storage_envelopes_up_1_dn_1,RTS-GMLC_envelope_benchmark_v2.1s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,549261.866100,OPTIMAL,549261.866100,demand_1,2,base_ramp_storage_envelopes_up_0_5_dn_0_5,RTS-GMLC_envelope_benchmark_v2.1s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,422630.533808,OPTIMAL,422630.533808,demand_1,3,base_ramp_storage_envelopes_up_1_dn_1,RTS-GMLC_envelope_benchmark_v2.1s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,428075.641942,OPTIMAL,428075.641942,demand_1,3,base_ramp_storage_envelopes_up_0_5_dn_0_5,RTS-GMLC_envelope_benchmark_v2.1s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,486029.065035,OPTIMAL,486029.065035,demand_1,2,base_ramp_storage_envelopes_up_1_dn_1,RTS-GMLC_envelope_compare_v3.1s,486029.065033,OPTIMAL,FEASIBLE_POINT,NO_SOLUTION,0.000000e+00,0.026007,FEASIBLE_POINT,FEASIBLE_POINT
1,549261.866100,OPTIMAL,549261.866100,demand_1,2,base_ramp_storage_envelopes_up_0_5_dn_0_5,RTS-GMLC_envelope_compare_v3.1s,549261.866100,OPTIMAL,FEASIBLE_POINT,NO_SOLUTION,2.119487e-16,0.029030,FEASIBLE_POINT,FEASIBLE_POINT
0,422630.533808,OPTIMAL,422630.533808,demand_1,3,base_ramp_storage_envelopes_up_1_dn_1,RTS-GMLC_envelope_compare_v3.1s,422630.533808,OPTIMAL,FEASIBLE_POINT,NO_SOLUTION,0.000000e+00,0.024827,FEASIBLE_POINT,FEASIBLE_POINT
1,428075.641942,OPTIMAL,428075.641942,demand_1,3,base_ramp_storage_envelopes_up_0_5_dn_0_5,RTS-GMLC_envelope_compare_v3.1s,428075.641939,OPTIMAL,FEASIBLE_POINT,NO_SOLUTION,0.000000e+00,0.028585,FEASIBLE_POINT,FEASIBLE_POINT
0,410310.535032,OPTIMAL,410310.535032,demand_1,2,base_ramp_storage_envelopes_up_1_dn_1,RTS-GMLC_e_reserve_benchmark_v2.0s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,435494.571559,OPTIMAL,435494.571559,demand_1,2,base_ramp_storage_envelopes_up_0_5_dn_0_5,RTS-GMLC_e_reserve_benchmark_v2.0s,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
gcdi_KPI_adequacy = gcdi_KPI_adequacy.merge(
    scalar[['solution_id', 'configuration', 'day', 'objective_value', 'iteration']],
    on=['solution_id', 'configuration', 'day','iteration'],
    suffixes=('', '_s')
)

In [44]:
filter_ = (gcdi_KPI_adequacy.iteration == 'demand_1') #&  (gcdi_KPI_adequacy.day == 3)
for k,v in gcdi_KPI_adequacy.loc[filter_,:].groupby(['day', 'µ', 'model_type']):
    for k2,v2 in v.groupby('model_type'):
        print("")
        print(f"Model Type: {k2}")
        print(v2[['objective_value_uc','objective_value', 'objective_value_s', 'day', 'µ', 'solution_id']])
    


Model Type: e-reserve
    objective_value_uc  objective_value  objective_value_s  day    µ  \
9        685489.038404    435494.571559      435494.571559    2  0.5   
13       685489.038368    435494.571559      435494.571559    2  0.5   

                           solution_id  
9   RTS-GMLC_e_reserve_benchmark_v2.0s  
13    RTS-GMLC_e_reserve_compare_v3.0s  

Model Type: envelope
   objective_value_uc  objective_value  objective_value_s  day    µ  \
1       685486.846716      549261.8661        549261.8661    2  0.5   
5       685486.846716      549261.8661        549261.8661    2  0.5   

                         solution_id  
1  RTS-GMLC_envelope_benchmark_v2.1s  
5    RTS-GMLC_envelope_compare_v3.1s  

Model Type: conservative
   objective_value_uc  objective_value  objective_value_s  day    µ  \
0       685486.846716    486029.065035      486029.065035    2  1.0   
4       685486.846716    486029.065035      486029.065035    2  1.0   

                         solution_id  
0  RT

In [45]:
gcdi_KPI_adequacy.iteration

0     demand_1
1     demand_1
2     demand_1
3     demand_1
4     demand_1
5     demand_1
6     demand_1
7     demand_1
8     demand_1
9     demand_1
10    demand_1
11    demand_1
12    demand_1
13    demand_1
14    demand_1
15    demand_1
Name: iteration, dtype: object

In [46]:
x = 552900.083994
y = 552900.083981
print( (x-y)/y)

2.3512354130043355e-11
